In [ ]:
import mdtraj as md

# Load topology and trajectory
topology = "stripped.pdb"
trajectory = "stripped.nc"

# Load trajectory with topology
traj = md.load(trajectory, top=topology)

# Select only C-alpha atoms
ca_indices = traj.topology.select("name CA")  # Select C-alpha atoms
traj_ca = traj.atom_slice(ca_indices)  # Extract C-alpha atoms

# Extract snapshots (frames)
num_frames = traj_ca.n_frames  # Total number of snapshots
print(f"Total snapshots: {num_frames}")

# Save first snapshot as PDB
traj_ca[0].save_pdb("first_snapshot.pdb")  # Saves the first frame with only C-alphas


In [1]:
import mdtraj as md
import pyemma.coordinates as coor
import numpy as np
import pandas as pd

# Load topology and trajectory
topology = "stripped.pdb"
trajectory = "stripped.nc"

# Load trajectory with topology
traj = md.load(trajectory, top=topology)

# Subsample every 50th frame
subsample_gap = 50
traj_subsampled = traj[::subsample_gap]  # Take every 50th frame

# Extract snapshots (frames) after subsampling
num_frames = traj_subsampled.n_frames  # Total number of snapshots after subsampling
print(f"Total snapshots after subsampling: {num_frames}")

# Save first subsampled snapshot as PDB
traj_subsampled[0].save_pdb("first_subsampled_snapshot.pdb")  # Save first frame

# Define feature for computing residue minimum distances
feat = coor.featurizer(topology)
feat.add_residue_mindist()  # Computes minimum distances between residues

# Compute residue minimum distances
def compute_residue_mindist(traj_object, top_file):
    try:
        # Save the subsampled trajectory
        subsampled_traj_file = "subsampled_traj.xtc"
        traj_object.save_xtc(subsampled_traj_file)  # Save subsampled trajectory
        
        # Create a source from the trajectory data
        source = coor.source(subsampled_traj_file, features=feat)
        
        # Load the computed minimum distance data
        data = source.get_output()
        
        return np.array(data)  # Return as numpy array
    
    except Exception as e:
        print(f"Error processing trajectory: {e}")
        return None

# Compute minimum distances using the full topology (not just C-alphas)
residue_mindist_array = compute_residue_mindist(traj_subsampled, topology)

# Print shape of computed distances
if residue_mindist_array is not None:
    print(f"Computed minimum distances shape: {residue_mindist_array.shape}")

residue_mindist_array = residue_mindist_array.reshape(2000,8911)
# Convert the numpy array to a DataFrame
df = pd.DataFrame(residue_mindist_array)

# Define the output file path
output_file = "42d_1.csv"
# Save the DataFrame to CSV
df.to_csv(output_file, index=False)
print(f"Results saved to {output_file}")


Total snapshots after subsampling: 2000
05-03-25 20:46:58 pyemma.coordinates.data.featurization.featurizer.MDFeaturizer[0] WARNING  Using all residue pairs with schemes like closest or closest-heavy is very time consuming. Consider reducing the residue pairs
Computed minimum distances shape: (1, 2000, 8911)
Results saved to 42d_1.csv


In [20]:
# Save the DataFrame to CSV
df.to_csv(output_file, index=False)
print(f"Results saved to {output_file}")


Results saved to 42d_1.csv
